In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.ecom_bronze;
CREATE SCHEMA IF NOT EXISTS workspace.ecom_silver;
CREATE SCHEMA IF NOT EXISTS workspace.ecom_gold;
CREATE SCHEMA IF NOT EXISTS workspace.ecom_quarantine;

In [0]:
from pyspark.sql import functions as F

VOLUME = "/Volumes/workspace/default/olist_raw"
BRONZE = "workspace.ecom_bronze"

TABLES = {
    "olist_customers_dataset.csv": "bronze_customers",
    "olist_orders_dataset.csv": "bronze_orders",
    "olist_order_items_dataset.csv": "bronze_order_items",
    "olist_order_payments_dataset.csv": "bronze_payments",
    "olist_order_reviews_dataset.csv": "bronze_reviews",
    "olist_products_dataset.csv": "bronze_products",
    "olist_sellers_dataset.csv": "bronze_sellers",
    "olist_geolocation_dataset.csv": "bronze_geolocation",
    "product_category_name_translation.csv": "bronze_category_translation",
}

for csv_name, table in TABLES.items():
    df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)     # reviews have newlines inside quoted text
        .option("escape", '"')         # ...and embedded quotes/commas
        .csv(f"{VOLUME}/{csv_name}")   # no inferSchema -> all strings (true bronze)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", lit(csv_name))
        .withColumn("source_system", lit("olist_batch"))
    )
    target = f"{BRONZE}.{table}"
    dwrite.format("delta").mode("overwrite").saveAsTable(target)
    print(f"{target:35s} rows={spark.table(target).count():,}")

workspace.ecom_bronze.bronze_customers rows=99,441
workspace.ecom_bronze.bronze_orders rows=99,441
workspace.ecom_bronze.bronze_order_items rows=112,650
workspace.ecom_bronze.bronze_payments rows=103,886
workspace.ecom_bronze.bronze_reviews rows=99,224
workspace.ecom_bronze.bronze_products rows=32,951
workspace.ecom_bronze.bronze_sellers rows=3,095
workspace.ecom_bronze.bronze_geolocation rows=1,000,163
workspace.ecom_bronze.bronze_category_translation rows=71


In [0]:
from pyspark.sql.functions import *

VALID_STATUSES = ["delivered", "shipped", "canceled", "unavailable",
                  "invoiced", "processing", "created", "approved"]

bronze = spark.table("workspace.ecom_bronze.bronze_orders")

# 1. TYPE: cast the raw string timestamps to real timestamps
typed = (
    bronze
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date"))
)

# 2. VALIDATE: build an array of error reasons (nulls = rule passed)
errors = array_compact(array(
    when(col("order_id").isNull(), "missing_order_id"),
    when(col("customer_id").isNull(), "missing_customer_id"),
    when(~col("order_status").isin(VALID_STATUSES), "invalid_order_status"),
    when((col("order_status") == "delivered")
           & col("order_delivered_customer_date").isNull(), "delivered_without_delivery_date"),
    when(col("order_approved_at") < col("order_purchase_timestamp"), "approved_before_purchase"),
    when(col("order_delivered_customer_date") < col("order_purchase_timestamp"), "delivered_before_purchase"),
))
validated = typed.withColumn("validation_errors", errors)

# 3. SPLIT: clean rows to silver, bad rows to quarantine with a reason
clean = validated.filter(size("validation_errors") == 0).drop("validation_errors")
quarantine = (validated.filter(size("validation_errors") > 0)
              .withColumn("error_timestamp", current_timestamp()))

clean.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.silver_orders")
quarantine.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_quarantine.quarantine_invalid_orders")

c, q = spark.table("workspace.ecom_silver.silver_orders").count(), spark.table("workspace.ecom_quarantine.quarantine_invalid_orders").count()
print(f"silver_orders: {c:,}   quarantine: {q:,}   total: {c+q:,} (should be 99,441)")

silver_orders: 99,433   quarantine: 8   total: 99,441 (should be 99,441)


In [0]:
display(
    spark.table("workspace.ecom_quarantine.quarantine_invalid_orders")
    .select(F.explode("validation_errors").alias("reason"))
    .groupBy("reason").count()
)

reason,count
delivered_without_delivery_date,8


In [0]:
from pyspark.sql.functions import *

bronze = spark.table("workspace.ecom_bronze.bronze_payments")

# 1. TYPE
typed = (
    bronze
    .withColumn("payment_sequential", col("payment_sequential").cast("int"))
    .withColumn("payment_installments", col("payment_installments").cast("int"))
    .withColumn("payment_value", col("payment_value").cast("double"))
)

# 2. VALIDATE the payment lines
errors = array_compact(array(
    when(col("order_id").isNull(), "missing_order_id"),
    when(col("payment_value") < 0, "negative_payment_value"),
    when(col("payment_type") == "not_defined", "undefined_payment_type"),
))
validated = typed.withColumn("validation_errors", errors)

# 3. SPLIT clean lines vs quarantine
clean = validated.filter(size("validation_errors") == 0).drop("validation_errors")
quarantine = (validated.filter(size("validation_errors") > 0)
              .withColumn("error_timestamp", current_timestamp()))

clean.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.silver_payments")
quarantine.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_quarantine.quarantine_invalid_payments")

# 4. AGGREGATE to ONE row per order — this is the step that protects revenue
order_payments = clean.groupBy("order_id").agg(
    round(sum("payment_value"), 2).alias("order_payment_total"),
    count("*").alias("payment_line_count"),
    countDistinct("payment_type").alias("distinct_payment_types"),
    max("payment_installments").alias("max_installments"),
    max(when(col("payment_type") == "voucher", 1).otherwise(0)).cast("boolean").alias("used_voucher"),
)
order_payments.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.silver_order_payments")

lines = spark.table("workspace.ecom_silver.silver_payments").count()
q = spark.table("workspace.ecom_quarantine.quarantine_invalid_payments").count()
orders = spark.table("workspace.ecom_silver.silver_order_payments").count()
print(f"silver_payments (lines): {lines:,}   quarantine: {q:,}   total: {lines+q:,} (should be 103,886)")
print(f"silver_order_payments (1 row/order): {orders:,}")

silver_payments (lines): 103,883   quarantine: 3   total: 103,886 (should be 103,886)
silver_order_payments (1 row/order): 99,437


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ---------- silver_geolocation: 1M rows -> one representative row per zip prefix ----------
geo = (spark.table("workspace.ecom_bronze.bronze_geolocation")
    .withColumn("geolocation_zip_code_prefix", col("geolocation_zip_code_prefix").cast("int"))
    .withColumn("geolocation_lat", col("geolocation_lat").cast("double"))
    .withColumn("geolocation_lng", col("geolocation_lng").cast("double"))
)

# representative coordinate per zip = average of all its points
coords = (geo.groupBy("geolocation_zip_code_prefix")
    .agg(round(avg("geolocation_lat"), 6).alias("lat"),
         round(avg("geolocation_lng"), 6).alias("lng")))

# most common city/state per zip, via row_number "pick the top per group"
w_city = Window.partitionBy("geolocation_zip_code_prefix").orderBy(desc("cnt"))
city_state = (geo.groupBy("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state")
    .agg(count("*").alias("cnt"))
    .withColumn("rn", row_number().over(w_city))
    .filter(col("rn") == 1)
    .select("geolocation_zip_code_prefix",
            col("geolocation_city").alias("city"),
            col("geolocation_state").alias("state")))

silver_geo = coords.join(city_state, "geolocation_zip_code_prefix")
silver_geo.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.silver_geolocation")

# ---------- silver_reviews: dedup 814 repeated review_ids, keep the latest ----------
reviews = (spark.table("workspace.ecom_bronze.bronze_reviews")
    .withColumn("review_score", col("review_score").cast("int"))
    .withColumn("review_creation_date", to_timestamp("review_creation_date"))
    .withColumn("review_answer_timestamp", to_timestamp("review_answer_timestamp")))

w_rev = Window.partitionBy("review_id").orderBy(desc("review_answer_timestamp"))
silver_reviews = (reviews
    .withColumn("rn", row_number().over(w_rev))
    .filter(col("rn") == 1)
    .drop("rn"))
silver_reviews.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.silver_reviews")

print("silver_geolocation (1 row/zip):", spark.table("workspace.ecom_silver.silver_geolocation").count())
print("silver_reviews (deduped):", spark.table("workspace.ecom_silver.silver_reviews").count(), "(should be 98,410)")

silver_geolocation (1 row/zip): 19015
silver_reviews (deduped): 98410 (should be 98,410)


In [0]:
from pyspark.sql.functions import *

# ---------- silver_customers: just typing ----------
(spark.table("workspace.ecom_bronze.bronze_customers")
    .withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("int"))
    .write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.silver_customers"))

# ---------- silver_sellers: just typing ----------
(spark.table("workspace.ecom_bronze.bronze_sellers")
    .withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("int"))
    .write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.silver_sellers"))

# ---------- silver_products: fix null categories, standardize names, add English category ----------
products = (spark.table("workspace.ecom_bronze.bronze_products")
    .withColumnRenamed("product_name_lenght", "product_name_length")          # source misspells "length"
    .withColumnRenamed("product_description_lenght", "product_description_length")
    .withColumn("product_name_length", col("product_name_length").cast("int"))
    .withColumn("product_description_length", col("product_description_length").cast("int"))
    .withColumn("product_photos_qty", col("product_photos_qty").cast("int"))
    .withColumn("product_weight_g", col("product_weight_g").cast("double"))
    .withColumn("product_length_cm", col("product_length_cm").cast("double"))
    .withColumn("product_height_cm", col("product_height_cm").cast("double"))
    .withColumn("product_width_cm", col("product_width_cm").cast("double"))
    .withColumn("product_category_name", coalesce(col("product_category_name"), lit("unknown"))))

# select ONLY the two needed cols from translation, or its metadata cols collide on join
translation = (spark.table("workspace.ecom_bronze.bronze_category_translation")
    .select("product_category_name", "product_category_name_english"))

(products.join(translation, "product_category_name", "left")
    .withColumn("product_category_name_english",
                coalesce(col("product_category_name_english"), col("product_category_name")))  # fallback for missing translations
    .write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.silver_products"))

# ---------- silver_order_items: typing + price/freight validation ----------
items = (spark.table("workspace.ecom_bronze.bronze_order_items")
    .withColumn("order_item_id", col("order_item_id").cast("int"))
    .withColumn("shipping_limit_date", to_timestamp("shipping_limit_date"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double")))

errors = array_compact(array(
    when(col("order_id").isNull(), "missing_order_id"),
    when(col("price") <= 0, "non_positive_price"),
    when(col("freight_value") < 0, "negative_freight"),
))
validated = items.withColumn("validation_errors", errors)
(validated.filter(size("validation_errors") == 0).drop("validation_errors")
    .write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.silver_order_items"))
(validated.filter(size("validation_errors") > 0).withColumn("error_timestamp", current_timestamp())
    .write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_quarantine.quarantine_invalid_order_items"))

for t in ["silver_customers", "silver_sellers", "silver_products", "silver_order_items"]:
    print(f"{t:22s}", spark.table(f"workspace.ecom_silver.{t}").count())
print("quarantine_order_items", spark.table("workspace.ecom_quarantine.quarantine_invalid_order_items").count())

silver_customers       99441
silver_sellers         3095
silver_products        32951
silver_order_items     112650
quarantine_order_items 0


In [0]:
from pyspark.sql.functions import *

geo = spark.table("workspace.ecom_silver.silver_geolocation")

# dim_customer — grain is customer_id, but we carry customer_unique_id for person-level CLV later
(spark.table("workspace.ecom_silver.silver_customers").alias("c")
    .join(geo.alias("g"), col("c.customer_zip_code_prefix") == col("g.geolocation_zip_code_prefix"), "left")
    .select("c.customer_id", "c.customer_unique_id", "c.customer_zip_code_prefix",
            "c.customer_city", "c.customer_state",
            col("g.lat").alias("customer_lat"), col("g.lng").alias("customer_lng"))
    .write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_gold.dim_customer"))

# dim_seller — grain is seller_id
(spark.table("workspace.ecom_silver.silver_sellers").alias("s")
    .join(geo.alias("g"), col("s.seller_zip_code_prefix") == col("g.geolocation_zip_code_prefix"), "left")
    .select("s.seller_id", "s.seller_zip_code_prefix", "s.seller_city", "s.seller_state",
            col("g.lat").alias("seller_lat"), col("g.lng").alias("seller_lng"))
    .write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_gold.dim_seller"))

# dim_product — grain is product_id
(spark.table("workspace.ecom_silver.silver_products")
    .select("product_id", "product_category_name", "product_category_name_english",
            "product_weight_g", "product_length_cm", "product_height_cm",
            "product_width_cm", "product_photos_qty")
    .write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_gold.dim_product"))

# dim_date — a calendar spine covering the order window
(spark.sql("SELECT explode(sequence(to_date('2016-09-01'), to_date('2018-12-31'), interval 1 day)) AS date_day")
    .withColumn("date_key", date_format("date_day", "yyyyMMdd").cast("int"))
    .withColumn("year", year("date_day"))
    .withColumn("quarter", quarter("date_day"))
    .withColumn("month", month("date_day"))
    .withColumn("month_name", date_format("date_day", "MMMM"))
    .withColumn("day", dayofmonth("date_day"))
    .withColumn("day_of_week", date_format("date_day", "EEEE"))
    .withColumn("is_weekend", dayofweek("date_day").isin(1, 7))
    .write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_gold.dim_date"))

for t in ["dim_customer", "dim_seller", "dim_product", "dim_date"]:
    print(f"{t:14s}", spark.table(f"workspace.ecom_gold.{t}").count())

dim_customer   99441
dim_seller     3095
dim_product    32951
dim_date       852


In [0]:
from pyspark.sql.functions import *

orders   = spark.table("workspace.ecom_silver.silver_orders")
payments = spark.table("workspace.ecom_silver.silver_order_payments")
items    = spark.table("workspace.ecom_silver.silver_order_items")

# roll item lines up to the order grain
item_agg = items.groupBy("order_id").agg(
    round(sum("price"), 2).alias("items_total"),
    round(sum("freight_value"), 2).alias("freight_total"),
    count("*").alias("item_count"))

fact_orders = (orders.alias("o")
    .join(payments.alias("p"), "order_id", "left")
    .join(item_agg.alias("i"), "order_id", "left")
    .select(
        col("order_id"),
        col("o.customer_id"),
        date_format("o.order_purchase_timestamp", "yyyyMMdd").cast("int").alias("order_date_key"),
        col("o.order_status"),
        col("o.order_purchase_timestamp"),
        col("o.order_delivered_customer_date"),
        col("o.order_estimated_delivery_date"),
        # payment measures — coalesce the ripple nulls to 0, but keep an honesty flag
        coalesce(col("p.order_payment_total"), lit(0.0)).alias("order_payment_total"),
        col("p.order_payment_total").isNotNull().alias("has_payment_info"),
        coalesce(col("p.payment_line_count"), lit(0)).alias("payment_line_count"),
        coalesce(col("p.max_installments"), lit(0)).alias("max_installments"),
        coalesce(col("p.used_voucher"), lit(False)).alias("used_voucher"),
        # item measures
        coalesce(col("i.items_total"), lit(0.0)).alias("items_total"),
        coalesce(col("i.freight_total"), lit(0.0)).alias("freight_total"),
        coalesce(col("i.item_count"), lit(0)).alias("item_count"),
        # delivery measures — leave NULL when not delivered (null is honest, 0 would lie)
        datediff(col("o.order_delivered_customer_date"), col("o.order_purchase_timestamp")).alias("delivery_days"),
        datediff(col("o.order_estimated_delivery_date"), col("o.order_purchase_timestamp")).alias("estimated_delivery_days"),
        datediff(col("o.order_delivered_customer_date"), col("o.order_estimated_delivery_date")).alias("delivery_delay_days"),
        (col("o.order_delivered_customer_date") > col("o.order_estimated_delivery_date")).alias("is_late")))

fact_orders.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_gold.fact_orders")

fo = spark.table("workspace.ecom_gold.fact_orders")
print("fact_orders rows:", fo.count())
print("orders missing payment info:", fo.filter(~col("has_payment_info")).count())
fo.agg(round(sum("order_payment_total"), 2).alias("total_revenue"),
       round(avg("delivery_days"), 1).alias("avg_delivery_days")).show()

fact_orders rows: 99433 (should be 99,433)
orders missing payment info: 4
+-------------+-----------------+
|total_revenue|avg_delivery_days|
+-------------+-----------------+
| 1.60074932E7|             12.5|
+-------------+-----------------+



In [0]:
from pyspark.sql.functions import *

items = spark.table("workspace.ecom_silver.silver_order_items")
orders = (spark.table("workspace.ecom_silver.silver_orders")
    .select("order_id",
            date_format("order_purchase_timestamp", "yyyyMMdd").cast("int").alias("order_date_key"),
            col("customer_id")))

fact_order_items = (items.alias("i")
    .join(orders.alias("o"), "order_id", "inner")     # inner = only items of clean orders
    .select(
        col("order_id"),
        col("i.order_item_id"),
        col("o.customer_id"),
        col("i.product_id"),
        col("i.seller_id"),
        col("o.order_date_key"),
        col("i.price"),
        col("i.freight_value"),
        round(col("i.price") + col("i.freight_value"), 2).alias("item_revenue"),
        col("i.shipping_limit_date")))

fact_order_items.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_gold.fact_order_items")

foi = spark.table("workspace.ecom_gold.fact_order_items")
print("fact_order_items rows:", foi.count())

# referential integrity check: every product_id and seller_id must exist in its dimension
prod_orphans = foi.join(spark.table("workspace.ecom_gold.dim_product"), "product_id", "left_anti").count()
sell_orphans = foi.join(spark.table("workspace.ecom_gold.dim_seller"), "seller_id", "left_anti").count()
print("product orphans:", prod_orphans, "  seller orphans:", sell_orphans)

fact_order_items rows: 112642 (just under 112,650)
product orphans: 0   seller orphans: 0
